<a href="https://colab.research.google.com/github/Manaspanwar20/RecommendationSystem/blob/main/recommendationsystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score, confusion_matrix,precision_score,recall_score
from sklearn.linear_model import  LogisticRegression

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
df_movies = pd.read_csv('/content/drive/MyDrive/movies.csv')
df_ratings = pd.read_csv('/content/drive/MyDrive/ratings.csv')
print("Movies loaded:", len(df_movies))
print("Ratings loaded:", len(df_ratings))

Movies loaded: 9742
Ratings loaded: 100836


In [12]:
user_movie_matrix = df_ratings.pivot(index='userId', columns='movieId', values='rating')
user_movie_matrix_filled = user_movie_matrix.fillna(0)
user_movie_matrix_filled.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
movie_user_matrix = user_movie_matrix_filled.T
movie_similarity = cosine_similarity(movie_user_matrix)
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)
movie_similarity_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.410562,0.296917,0.035573,0.308762,0.376316,0.277491,0.131629,0.232586,0.395573,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.410562,1.000000,0.282438,0.106415,0.287795,0.297009,0.228576,0.172498,0.044835,0.417693,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.296917,0.282438,1.000000,0.092406,0.417802,0.284257,0.402831,0.313434,0.304840,0.242954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.035573,0.106415,0.092406,1.000000,0.188376,0.089685,0.275035,0.158022,0.000000,0.095598,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.308762,0.287795,0.417802,0.188376,1.000000,0.298969,0.474002,0.283523,0.335058,0.218061,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
def recommend_similar_movies(movie_title_keyword, num_recommendations=5):
    match = df_movies[df_movies['title'].str.contains(movie_title_keyword, case=False)]
    if len(match) == 0:
        return "Movie not found."
    matched_movie = match.iloc[0]
    movie_id = matched_movie['movieId']
    full_title = matched_movie['title']

    similarity_scores = movie_similarity_df[movie_id]
    top_movie_ids = similarity_scores.sort_values(ascending=False)[1:num_recommendations+1].index

    recommended_movies = df_movies[df_movies['movieId'].isin(top_movie_ids)]

    print(f"Recommendations based on: {full_title}")
    for idx, row in recommended_movies.iterrows():
        print(f"{row['title']} | Genres: {row['genres']}")

recommend_similar_movies("Finding Nemo")
print()
recommend_similar_movies("Batman")

Recommendations based on: Finding Nemo (2003)
Shrek (2001) | Genres: Adventure|Animation|Children|Comedy|Fantasy|Romance
Monsters, Inc. (2001) | Genres: Adventure|Animation|Children|Comedy|Fantasy
Pirates of the Caribbean: The Curse of the Black Pearl (2003) | Genres: Action|Adventure|Comedy|Fantasy
Shrek 2 (2004) | Genres: Adventure|Animation|Children|Comedy|Musical|Romance
Incredibles, The (2004) | Genres: Action|Adventure|Animation|Children|Comedy

Recommendations based on: Batman Forever (1995)
Ace Ventura: Pet Detective (1994) | Genres: Comedy
True Lies (1994) | Genres: Action|Adventure|Comedy|Romance|Thriller
Cliffhanger (1993) | Genres: Action|Adventure|Thriller
Dances with Wolves (1990) | Genres: Adventure|Drama|Western
Batman (1989) | Genres: Action|Crime|Thriller
